In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# 전체 CSV 로딩
volume_path = "/Volumes/bronze_api/agrofood_perday/volumn/*.csv"
df = spark.read.option("header", True).option("inferSchema", True).csv(volume_path)

print(f"행 개수: {df.count():,}")
print(f"컬럼 개수: {len(df.columns)}")
print(f"\n컬럼 목록:")
for c in df.columns:
    print(f"  - {c}")


# 기초통계 (수치형)
# display(df.select(price_cols).describe())


# # 가격 관련 컬럼 숫자형으로 변환
# price_cols = [c for c in df.columns if 'prc' in c]
# numeric_cols = ['ctgry_cd', 'grd_cd', 'item_cd', 'se_cd', 'unit_sz', 'vrty_cd'] + price_cols

# df2 = df
# for col_name in numeric_cols:
#     df2 = df2.withColumn(col_name, F.col(col_name).cast(DoubleType()))

In [0]:
cat_cols = ['ctgry_nm', 'item_nm', 'vrty_nm', 'grd_nm', 'se_nm', 'sgg_nm', 'mrkt_nm','unit']
print("범주형 컬럼 고유값 수")
for c in cat_cols:
    cnt = df.select(c).distinct().count()
    print(f"  {c}: {cnt}개")

display(df.groupBy('ctgry_cd','ctgry_nm').count().orderBy(F.desc('count'))) # 부류명
display(df.groupBy('item_cd','item_nm').count().orderBy(F.desc('count')))  # 품목명
display(df.groupBy('vrty_cd','vrty_nm').count().orderBy(F.desc('count')))  # 품종명
display(df.groupBy('grd_cd','grd_nm').count().orderBy(F.desc('count')))   # 등급명
display(df.groupBy('se_cd','se_nm').count().orderBy(F.desc('count')))     # 구분명
display(df.groupBy('sgg_cd','sgg_nm').count().orderBy(F.desc('count')))   # 시군구명
display(df.groupBy('mrkt_cd','mrkt_nm').count().orderBy(F.desc('count')))   # 시장명
display(df.groupBy('unit','unit_sz').count().orderBy(F.desc('count')))     # 단위

In [0]:
# 일별 도·소매 가격정보 기준 컬럼 한글명 매핑
col_kor_map = {
    "exmn_ymd": "조사일자",
    "sgg_cd": "시군구코드",
    "sgg_nm": "시군구명",
    "se_cd": "구분코드",
    "se_nm": "구분명",
    "ctgry_cd": "부류코드",
    "ctgry_nm": "부류명",
    "item_cd": "품목코드",
    "item_nm": "품목명",
    "vrty_cd": "품종코드",
    "vrty_nm": "품종명",
    "grd_cd": "등급코드",
    "grd_nm": "등급명",
    "unit": "단위",
    "unit_sz": "단위크기",
    "mrkt_cd": "시장코드",
    "mrkt_nm": "시장명",
    "exmn_dd_prc": "조사일가격",
    "exmn_dd_cnvs_prc": "조사일kg환산가격",
    "orgnl_reg_dt": "원본등록일시"
}

total = df.count()

notnull_counts = df.select(
    [
        F.sum(F.when(F.col(c).isNotNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]
).toPandas().T

notnull_counts.columns = ['notnull_count']
notnull_counts['notnull_pct'] = (notnull_counts['notnull_count'] / total * 100).round(2)
notnull_counts['null_count'] = total - notnull_counts['notnull_count']
notnull_counts['null_pct'] = (notnull_counts['null_count'] / total * 100).round(2)
notnull_counts['col_name'] = notnull_counts.index
notnull_counts['col_kor'] = notnull_counts['col_name'].map(col_kor_map).fillna("")

notnull_counts = notnull_counts[['col_name', 'col_kor', 'notnull_count', 'null_count', 'null_pct']]

display(notnull_counts)